In [ ]:
import os
import numpy as np
import pandas as pd
import ast
import time
import tracemalloc
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed
from sklearn.metrics import precision_score, recall_score, f1_score
import shap
import warnings
warnings.filterwarnings('ignore') # Hides annoying SHAP warnings

# --- Sanity Check for ALL files ---
# 1. Check if the master label file exists
if not os.path.exists('labeled_anomalies.csv'):
    print("❌ Error: 'labeled_anomalies.csv' is missing. Check your root folder.")
else:
    # 2. Read the CSV to get the list of all expected channels
    labels_df = pd.read_csv('labeled_anomalies.csv')
    channels = labels_df['chan_id'].unique()
    
    missing_files = []
    
    # 3. Check train and test folders for every single channel listed in the CSV
    for chan in channels:
        # Note: Using 'data/data/' based on your previous code snippet
        train_path = f'data/data/train/{chan}.npy'
        test_path = f'data/data/test/{chan}.npy'
        
        if not os.path.exists(train_path):
            missing_files.append(train_path)
        if not os.path.exists(test_path):
            missing_files.append(test_path)
            
    # 4. Final Verification
    if len(missing_files) == 0:
        print(f"Success: Labels and all {len(channels)} channels (train & test data) found! Environment is ready.")
    else:
        print(f"❌ Error: Missing {len(missing_files)} files.")
        print(f"For example, couldn't find: {missing_files[:3]}") # Prints the first 3 missing files so you know where to look

✅ Success: Labels and all 81 channels (train & test data) found! Environment is ready.


In [ ]:
# --- CONFIG ---
channel = 'P-1' 

print(f"Loading telemetry for channel: {channel}...")
X_train = np.load(f'data/data/train/{channel}.npy')
X_test = np.load(f'data/data/test/{channel}.npy')

# --- PARSE LABELS ---
labels_df = pd.read_csv('labeled_anomalies.csv')
channel_info = labels_df[labels_df['chan_id'] == channel].iloc[0]
anomaly_sequences = ast.literal_eval(channel_info['anomaly_sequences'])

# Create ground truth array
y_test = np.zeros(len(X_test))
for seq in anomaly_sequences:
    start, end = seq
    y_test[start:end] = 1

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Loading telemetry for channel: A-7...
Training data shape: (2879, 25)
Testing data shape: (8631, 25)


In [ ]:
def run_zscore(train_data, test_data, threshold=2.0):
    start_time = time.time()
    tracemalloc.start()
    
    # Feature 0 is the actual sensor reading, the rest are encoded commands
    train_val = train_data[:, 0]
    test_val = test_data[:, 0]
    
    mean, std = np.mean(train_val), np.std(train_val)
    z_scores = np.abs((test_val - mean) / std)
    predictions = (z_scores > threshold).astype(int)
    
    _, peak_ram = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    inf_time = (time.time() - start_time) * 1000
    ram_mb = peak_ram / (1024 * 1024)
    return predictions, inf_time, ram_mb

print("Running Z-Score Baseline...")
z_preds, z_time, z_ram = run_zscore(X_train, X_test)
print("Z-Score complete.")

Running Z-Score Baseline...
✅ Z-Score complete.


In [ ]:
# Prepare sequences (lookback of 50 timesteps to capture context)
def to_sequences(data, seq_size=50):
    x = []
    for i in range(len(data) - seq_size):
        x.append(data[i:(i + seq_size)])
    return np.array(x)

X_train_seq = to_sequences(X_train)
X_test_seq = to_sequences(X_test)
y_test_seq = y_test[50:] # Match label length to sequence length

print("Building and training LSTM Autoencoder...")
model = Sequential([
    LSTM(64, activation='relu', input_shape=(50, X_train.shape[1]), return_sequences=False),
    RepeatVector(50),
    LSTM(64, activation='relu', return_sequences=True),
    TimeDistributed(Dense(X_train.shape[1]))
])

model.compile(optimizer='adam', loss='mse')
# Training the model
model.fit(X_train_seq, X_train_seq, epochs=5, batch_size=64, validation_split=0.1, verbose=1)

# Inference (Testing)
print("Running LSTM Inference...")
start_time = time.time()
tracemalloc.start()

reconstructed = model.predict(X_test_seq)
mse = np.mean(np.power(X_test_seq - reconstructed, 2), axis=(1,2))

# Dynamic thresholding: flag top 5% highest errors as anomalies
threshold_lstm = np.percentile(mse, 95) 
lstm_preds = (mse > threshold_lstm).astype(int)

_, peak_ram_lstm = tracemalloc.get_traced_memory()
tracemalloc.stop()
lstm_time = (time.time() - start_time) * 1000
print("LSTM complete.")

Building and training LSTM Autoencoder...
Epoch 1/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - loss: 0.0083 - val_loss: 0.0074
Epoch 2/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0070 - val_loss: 0.0070
Epoch 3/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.0068 - val_loss: 0.0069
Epoch 4/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0067 - val_loss: 0.0069
Epoch 5/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0067 - val_loss: 0.0068
Running LSTM Inference...
269/269 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step
✅ LSTM complete.


In [ ]:
import shap
import numpy as np

print("Calculating SHAP values (using Model-Agnostic approach)...")

# 1. SHAP requires 2D inputs for this method. We create a wrapper function 
# that takes flat 2D data, reshapes it for the LSTM, and returns the anomaly score.
def lstm_anomaly_scorer(flattened_data):
    # Reshape from 2D back to 3D: (batch_size, 50 timesteps, features)
    reshaped_data = flattened_data.reshape(-1, 50, X_train.shape[1])
    
    # Get LSTM reconstructions
    reconstructed = model.predict(reshaped_data, verbose=0)
    
    # Calculate the Mean Squared Error (The Anomaly Score)
    mse = np.mean(np.power(reshaped_data - reconstructed, 2), axis=(1, 2))
    return mse

# 2. Flatten a small background dataset to 2D
background_3d = X_train_seq[:25] # Kept very small to save compute time
background_2d = background_3d.reshape(background_3d.shape[0], -1)

# 3. Initialize KernelExplainer (Avoids TensorFlow graph errors entirely)
explainer = shap.KernelExplainer(lstm_anomaly_scorer, background_2d)

# 4. Explain the first test sequence
sample_3d = X_test_seq[0:1]
sample_2d = sample_3d.reshape(1, -1)

print("Running explainer (this may take 10-30 seconds)...")
# We suppress warnings here as SHAP likes to complain about performance
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    shap_values = explainer.shap_values(sample_2d)

print("SHAP calculated successfully.")


Calculating SHAP values (using Model-Agnostic approach)...
Running explainer (this may take 10-30 seconds)...


100%|██████████| 1/1 [00:12<00:00, 12.80s/it]

✅ SHAP calculated successfully.


In [19]:
# Calculate total sequences for per-sequence latency
num_seq = len(X_test_seq)

results = pd.DataFrame({
    "Method": ["Z-Score", "LSTM AE"],
    "Precision": [precision_score(y_test, z_preds, zero_division=0), precision_score(y_test_seq, lstm_preds, zero_division=0)],
    "Recall": [recall_score(y_test, z_preds, zero_division=0), recall_score(y_test_seq, lstm_preds, zero_division=0)],
    "F1 Score": [f1_score(y_test, z_preds, zero_division=0), f1_score(y_test_seq, lstm_preds, zero_division=0)],
    "Inf Time/Seq (ms)": [z_time/len(X_test), lstm_time/num_seq],
    "Peak RAM (MB)": [z_ram, peak_ram_lstm/(1024*1024)]
})

# Display the clean table
display(results)

,Method,Precision,Recall,F1 Score,Inf Time/Seq (ms),Peak RAM (MB)
0,Z-Score,0.000000,0.000000,0.000000,0.000000,0.140639
1,LSTM AE,0.123543,0.022083,0.037469,0.462352,206.496777


The Z-score baseline doesn't perform very well. It hsa very low precision and recall, indicating it struggles to capture complex temporal patterns in the telemetry and produces both false positives and missed anomalies. The LSTM autoencode improves performance slightly, achieving higher precision, recall and F1 score. This suggests that is it better at modeling sequentiall dependencies and subtle deviations in the data. However, it has a much higher inference time and memeory usage, making it less suitable for strict real-time or resoruce-constrained environments. The Z-score tends to catch larger, more obvious deviations while missing the smaller anomaleis. On the other hand, the LSTM can detect the more subtle anomalies but still lacks strong overall accuracy. Overall, there is a trade-off between the simplicity and efficiency of the Z-score and the modeling power of the LSTM autoencoder. 